VALIDACIÓN DEL MÉTODO DE PRONÓSTICO HIDROLÓGICO ESP

Este código utiliza el modelo de Temez y datos de precipitación y evapotranspiración del período histórico de 1990 a 2020 para generar ensambles de pronóstico hidrológico considerando cada año de la serie histórica como forzante climática.

In [2]:
#importacion
from datetime import datetime, timedelta
import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from TemezRegional import TemezRegional

In [2]:
# Definiciones

lead_time = 6
start_year = 1985
end_year = 2025

#vamos a empezar un conteo para registrar errores al calcular ensambles, bastante más adelante en el código:
errores = 0

In [3]:
# Cargar datos

precip = pd.read_csv(
    "../../01_Datos_entrada/Pmedias.csv",
    header=None
)

etp = pd.read_csv(
    "../../01_Datos_entrada/ETPmedias.csv",
    header=None
)

ad = pd.read_csv(
    "../../01_Datos_entrada/AguaDisponible.txt",
    delim_whitespace=True,
    header=None
)

cuencas_n3 = pd.read_csv(
    "../../01_Datos_entrada/cuenca_nivel3.csv",
    header=None
)

In [4]:
# Crear lista de cuencas n2 a partir de las cuencas n3:

codigos_n3 = cuencas_n3.iloc[0, :].to_numpy().astype(int)
codigos_n2 = np.unique((codigos_n3 // 10).astype(int))

Acá abajo vamos a processar los datos para que tengan fecha como index y cuenca n3 como nombre de las columnas

In [5]:
#Separar metadatos

meta_precip = precip.iloc[0, :]
meta_etp = etp.iloc[0, :]
# los datos reales estan a partir de la fila 1
precip_data = precip.iloc[1:, :].copy()
etp_data = etp.iloc[1:, :].copy()

# Crear fechas (datetime)
dates = pd.to_datetime(
    dict(
        year=precip_data.iloc[:, 0].astype(int),
        month=precip_data.iloc[:, 1].astype(int),
        day=15
    )
)

dates = pd.DatetimeIndex(dates)

# Quitar año y mes de las séries
precip_data = precip_data.iloc[:, 2:].copy()
etp_data = etp_data.iloc[:, 2:].copy()

# Asignar fecha (datetime) como índice
precip_data.index = dates
etp_data.index = dates
precip_data.index.name = "time"
etp_data.index.name = "time"

# Nombrar las columnas de las series con la cuenca correspondiente
precip_data.columns = codigos_n3
etp_data.columns    = codigos_n3

#agregar index a ad y transformarlo en una serie
ad_series = pd.Series(
    ad.iloc[:, 0].values,
    index=codigos_n3,
    name="n3"
)

#print(ad_series)
#print(precip_data)
#print(etp_data)
#acá tenemos las séries históricas con index fecha y nombre de columna de la cuenca correspondiente.

Acá empieza un loop para correr el modelo para cada cuenca n3 y guardar el caudal y los estados internos para cada paso de tiempo.

In [6]:
#Empezamos creando contenedores
QC_historica = {} #contenedor para el caudal modelado con clima hstórico
states_internal_status = {} #contendor para los estados internos para cada paso de tiempo

for cuenca_n3 in precip_data.columns:

    P   = precip_data[cuenca_n3].to_numpy() # devuelve array numérico (série de precipitacion)
    ETP = etp_data[cuenca_n3].to_numpy() # devuelve array numérico (série de evapotranspiracion)
    AD = ad_series.loc[cuenca_n3] #.loc devuelve los valores de la etiquita cuenca.n3 (el de ad para esta cuenca)

    #correr el modelo de temez con los datos recien extraidos
    fluxes, states = TemezRegional(
        AD=AD,
        P=P,
        ETP=ETP
    )

    QC_historica[cuenca_n3] = pd.Series(fluxes["QC"], index=precip_data.index) #guardar caudal en el contenedor

    states_internal_status[cuenca_n3] = { #guardar estados internos en el contenedor
        "H": pd.Series(states["H"], index=precip_data.index),
        "V": pd.Series(states["V"], index=precip_data.index)
    }

#print (QC_historica)
#print (states_internal_status)

Hasta ahora corremos el modelo con todos los datos históricos disponibles, así tenemos los estados más calientes posibles.
Para empezar a trabajar con la matriz de ensambles, antes vamos filtrar las variables climaticas para el período deseado:

In [7]:
# Filtrar período
precip_f = precip_data.loc[str(start_year):str(end_year)]
etp_f    = etp_data.loc[str(start_year):str(end_year)]
dates_f  = precip_f.index
#print(precip_cuenca_f)
#print(dates_f)

#esto acá abajo despues se utiliza para garantinzar que no se intente correr ensambles con fechas fuera de rango
fecha_min_datos = precip_f.index.min()
fecha_max_datos = precip_f.index.max()

Ahora ya podemos empezar a calcular los ensambles. Lo que queremos es correr cada secuencia climatica histórica (ensamble) para cada cuenca con los estados iniciales de cada fecha de emisión. La extructura queda así:
Para cada "fecha de emisión" (t0) dentro del rango historico (estados inciales)
    para cada cuenca nivel 3 (n3) dentro de todas las cuencas nivel 3 (AD, eto, precip)
        para cada ano disponible en la série histórica que no sea el ano de emisión (ano) correr el modelo

In [8]:
#contenedor para los resultados
results_esp_n3 = {}

#las fechas de emisión del pronóstico son el rango histórico - el lead time
fechas_emision = dates_f[:-lead_time]
#lista de años para buscar los ensambles
anos_hist = np.unique(dates_f.year)

In [9]:
#primer bucle para cada fecha de emisión
for t0 in tqdm(fechas_emision, desc="Procesando ESP"):

    #selecciona el año de t0 y localiza su index para posteriormente poder filtrar de la lista de ensambles
    y0 = t0.year
    i0 = dates_f.get_loc(t0)

    #obtiene las fechas de pronóstico del esp (el horizonte de pronóstico)
    #dates_f tiene que ser fechas ordenadas y contínuas, porque future dates se selecciona a partir del indice de dates_f
    future_dates = dates_f[i0 + 1 : i0 + 1 + lead_time]

    #creamos un contenedor para cada fecha de emisión dentro del contenedor de resultados
    results_esp_n3[t0] = {}

    #-------- Ahora vamos para el próximo nivel de loop, por cuenca n3. ---------------#
    for cuenca_n3 in precip_f.columns:

        #para cada cuenca de nivel 3 queremos tener los estados calientes
        H0 = states_internal_status[cuenca_n3]["H"].loc[t0]
        V0 = states_internal_status[cuenca_n3]["V"].loc[t0]
    
        #también filtramos precip y evapo:
        precip_cuenca_f = precip_f[cuenca_n3]
        etp_cuenca_f = etp_f[cuenca_n3]
    
        #y también ad
        ad_cuenca = ad_series.loc[cuenca_n3]
    
        #creamos un contenedor para cada cuenca n3 dentro del contendor de fecha de emisión
        results_esp_n3[t0][cuenca_n3] = {}
    
        #print(precip_cuenca_f)

        #-------- Ahora vamos empezar con el loop que corre el modelo para cada año climático a partir
        #de los estados calientes de cada cada cuenca n3 para cada fecha de emisión (t0). -----------#

        for ano in anos_hist:

            #filtrar el año de fecha de emisión de los ensambles:
            if ano == y0:
                continue
        
            #ahora vamos crear las fechas de este ensamble. Se calcula la diferencia de años entre el "ano" del bucle y "y0", luego se aplica
            #esta diferencia a las fechas de future_dates.
            ens_dates = [
                d + pd.DateOffset(years=(ano - y0))
                for d in future_dates
            ]

            #acá agregamos una verficación para no intentar crear ensambles en meses sin datos
            if (min(ens_dates) < fecha_min_datos) or (max(ens_dates) > fecha_max_datos):
                continue
        
            #ahora vamos a seleccionar los valores de precipitacion y evapo para este ensamble
            try:
                P_esp = precip_cuenca_f.loc[ens_dates].to_numpy()
                ETP_esp = etp_cuenca_f.loc[ens_dates].to_numpy()
            except KeyError:
                print(f"Faltan datos para año {ano}, cuenca {cuenca_n3}")
                errores += 1
                continue
        
            #finalmente corremos el modelo para cada ensamble
            fluxes, states = TemezRegional(
                AD = ad_cuenca,
                P = P_esp,
                ETP = ETP_esp,
                H0 = H0,
                V0 = V0,
            )
        
            QC = fluxes["QC"]
        
            results_esp_n3[t0][cuenca_n3][ano] = QC
        
        #print (results_esp_n3)
                
    

Procesando ESP: 100%|██████████████████████████████████████████████████████████████| 486/486 [3:34:34<00:00, 26.49s/it]


Ahora que ya realizamos la corrida histórica del modelo para cuencas n3 y ya calculamos todos los ensambles para cuencas n3, vamos agregar los resultados a cuencas n2, empezando por la corrida histórica.

In [10]:
#contenedor
results_hist_n2 = {}

#convertir QC_historica a un dataframe
QC_hist_df = pd.DataFrame(QC_historica)
QC_hist_df.index.name = "time"
#print(QC_hist_df[101])

#filtrar fechas para el período histórico que queremos trabajar
QC_hist_df_f = QC_hist_df.loc[dates_f]
#print(QC_hist_df_f)

#loop para agregar los valores a cada cuenca n2
for cuenca_deseada in codigos_n2:
    
    #filtrar las cuencas n3 que pertenecen a la n2 del loop
    columnas_n3 = [c for c in QC_hist_df_f.columns if c // 10 == cuenca_deseada]
    #print(QC_hist_df_n2)
    
    #sumar los valores de todas las n3
    QC_hist_n2 = QC_hist_df_f[columnas_n3].sum(axis=1)
    QC_hist_n2.name = "QC"

    #organizar el dataframe para que sea compatible con el dataframe de salida del esp
    df_temp = pd.DataFrame({
        "fecha_pronostico": QC_hist_n2.index,
        "QC_hist": QC_hist_n2.values,
        "lead": 0
    })
    
    # guardar
    results_hist_n2[cuenca_deseada] = df_temp

    
#print (results_hist_n2)

ahora vamos acumular a n2 los resultados del esp. Venimos desde una estructura que es: - para cada fecha de emisión (t0) -> todas las cuencas n3 y para cada cuenca n3 -> cada ensamble calculado

In [ ]:
#empezamos el loop por cuenca n2
for cuenca_deseada in codigos_n2:

    # contenedor para la cuenca del loop
    results_cuenca = {}

    #empezamos a acceder a la extructura de los datos de results_esp_n3
    for t0 in results_esp_n3:
        results_cuenca[t0] = {} #contenedor

        for cuenca_n3 in results_esp_n3[t0]:
            
            #acá ya filtramos las cuencas n3 que pertenecen a la cuenca n2 del loop
            if cuenca_n3 // 10 != cuenca_deseada:
                continue

            for ano, QC in results_esp_n3[t0][cuenca_n3].items(): #accedemos a las los leadtimes por medio de .items

                #este bucle copia los valores de cuenca n3 si es la primera del bucle, de la segunda en adelante empieza a sumar
                if ano not in results_cuenca[t0]: 
                    results_cuenca[t0][ano] = QC.copy()
                else:
                    results_cuenca[t0][ano] += QC 
    #print(results_cuenca)


    # vamos pasar estos datos a un DataFrame

    rows = [] #creamos una lista contenedor

    #ahora vamos reorganizar los resultados
    for t0, ensambles in results_cuenca.items():
        for ano, QC in ensambles.items():
            for k, q in enumerate(QC, start=1):
                rows.append({
                    "cuenca_n2": cuenca_deseada,
                    "fecha_emision": t0,
                    "ano": ano,
                    "lead": k,
                    "fecha_pronostico": t0 + pd.DateOffset(months=k),
                    "QC": q
                })

    df_esp = pd.DataFrame(rows, columns=[
        "cuenca_n2",
        "fecha_emision",
        "ano",
        "lead",
        "fecha_pronostico",
        "QC"
    ])

    #ya tenemos los datos en dataframe, seguir organizando:
    
    df_esp_wide = (
        df_esp
        .pivot( #cambiar de columnas a filas
            index=["fecha_emision", "lead", "fecha_pronostico"],
            columns="ano",
            values="QC"
        )
        .reset_index()
    )

    #nombrar los ensanbles por año y organizarlos en columnas
    df_esp_wide.columns = [
        f"ens_{int(c)}" if isinstance(c, (int, np.integer)) else c
        for c in df_esp_wide.columns
    ]

    #el dataframe final debe tener los valores de los ensambles y de qc_hist también

    df_hist_cuenca = results_hist_n2[cuenca_deseada]
    
    df_final = (
        df_esp_wide
        .merge(
            df_hist_cuenca[["fecha_pronostico", "QC_hist"]], # <--- Usar el específico
            on="fecha_pronostico",
            how="left"
        )
        .sort_values(["fecha_emision", "lead"])
        .reset_index(drop=True)
    )

    cols_base = ["fecha_emision", "lead", "fecha_pronostico", "QC_hist"] #columnas iniciales
    cols_ens = sorted(c for c in df_final.columns if c.startswith("ens_")) #columnas de los ensambles

    df_final["ens_prom"] = df_final[cols_ens].mean(axis=1) #crear una columna con el promedio de los ensambles
    df_final = df_final[cols_base + ["ens_prom"] + cols_ens] #juntar todo

    #exportar
    df_final.to_csv(
        rf"../Resultados/01_Tablas_de_resultados/{cuenca_deseada}_df_esp_con_historico.csv",
        index=False,
        float_format="%.3f"
    )

Explorar resultados.
En la proxima celda podes crear un gráfico clásico del esp para una cuenca específica en una fecha específica.

In [ ]:
#primero seleccionar cuenca
cuenca_sel = 60
#seleccionar fecha de emisión del pronóstico
fecha_emision_sel = pd.Timestamp("2018-06-15") #YYYY-MM-15

# rango temporal del gráfico
start_date_chart = fecha_emision_sel - timedelta(days=365)
end_date_chart = fecha_emision_sel + pd.DateOffset(months=lead_time)
fechas_full = pd.date_range(start=start_date_chart, end=end_date_chart, freq="MS")

# ==============================
# HISTÓRICO → percentiles mensuales
# ==============================
corrida_hist = results_hist_n2[cuenca_sel].copy()

corrida_hist["mes"] = corrida_hist["fecha_pronostico"].dt.month

stats_hist = (
    corrida_hist
    .groupby("mes")["QC_hist"]
    .quantile([0.10, 0.25, 0.75, 0.90])
    .unstack()
    .rename(columns={0.10:"p10", 0.25:"p25", 0.75:"p75", 0.90:"p90"})
    .reset_index()
)

# serie histórica para línea negra
corrida_hist = corrida_hist.set_index("fecha_pronostico")
QC_chart = corrida_hist.loc[start_date_chart:end_date_chart, "QC_hist"]
QC_chart_max = QC_chart.max()

# ==============================
# ESP → percentiles de ensamble
# ==============================
path = rf"../Resultados/01_Tablas_de_resultados/{cuenca_sel}_df_esp_con_historico.csv"
df = pd.read_csv(path, parse_dates=["fecha_emision", "fecha_pronostico"])

df_sel = df[df["fecha_emision"] == fecha_emision_sel].copy()

cols_ens = [c for c in df_sel.columns if c.startswith("ens_") and c != "ens_prom"]

df_sel["p10_ens"] = df_sel[cols_ens].quantile(0.10, axis=1)
df_sel["p25_ens"] = df_sel[cols_ens].quantile(0.25, axis=1)
df_sel["p50_ens"] = df_sel[cols_ens].quantile(0.50, axis=1)
df_sel["p75_ens"] = df_sel[cols_ens].quantile(0.75, axis=1)
df_sel["p90_ens"] = df_sel[cols_ens].quantile(0.90, axis=1)

# valor observado en fecha de emisión
q_obs_ini = corrida_hist.loc[fecha_emision_sel, "QC_hist"]
fecha_ini = fecha_emision_sel

# crear fila inicial
fila_ini = pd.DataFrame({
    "fecha_pronostico": [fecha_ini],
    "p10_ens": [q_obs_ini],
    "p25_ens": [q_obs_ini],
    "p50_ens": [q_obs_ini],
    "p75_ens": [q_obs_ini],
    "p90_ens": [q_obs_ini],
})

df_sel = pd.concat([fila_ini, df_sel], ignore_index=True)

# ordenar por fecha (por seguridad)
df_sel = df_sel.sort_values("fecha_pronostico")

#print(df_sel)

# ==============================
# DATAFRAME FINAL PARA PLOT
# ==============================
df_plot = pd.DataFrame({"fecha_pronostico": fechas_full})

# agregar mes
df_plot["mes"] = df_plot["fecha_pronostico"].dt.month

# agregar climatología (todo el eje)
df_plot = df_plot.merge(stats_hist, on="mes", how="left")

#print(df_plot)

NameError: name 'corrida_histórica' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
ax.plot(QC_chart.index, QC_chart.values,
        color='black', linewidth=2, marker='o',
        mfc='white', mec='k', label='_nolegend_')

ax.fill_between(df_plot['fecha_pronostico'], 0, df_plot['p10'], color='#CD233F', alpha=0.3,label="Bajo")
ax.fill_between(df_plot['fecha_pronostico'], df_plot['p10'], df_plot['p25'], color='#FFA885', alpha=0.3,label="Inferior a lo normal")
ax.fill_between(df_plot['fecha_pronostico'], df_plot['p25'], df_plot['p75'], color='#E7E2BC', alpha=0.3,label="Normal")
ax.fill_between(df_plot['fecha_pronostico'], df_plot['p75'], df_plot['p90'], color='#8ECEEE', alpha=0.3,label="Superior a lo normal")
ax.fill_between(df_plot['fecha_pronostico'], df_plot['p90'], 2000,color='#2C7DCD', alpha=0.3,label="Alto")


ax.plot(df_sel['fecha_pronostico'],df_sel['p10_ens'],color = 'red',linestyle = '--',linewidth=2, mec='k', label='Percentil 10')
ax.plot(df_sel['fecha_pronostico'],df_sel['p25_ens'],color = 'orange',linestyle = '--',linewidth=2, mec='k', label='Percentil 25')
ax.plot(df_sel['fecha_pronostico'],df_sel['p50_ens'],color = 'blue',linestyle = '--',linewidth=2, mec='k', label='Percentil 50')
ax.plot(df_sel['fecha_pronostico'],df_sel['p75_ens'],color = 'gray',linestyle = '--',linewidth=2, mec='k', label='Percentil 75')
ax.plot(df_sel['fecha_pronostico'],df_sel['p90_ens'],color = 'black',linestyle = '--',linewidth=2, mec='k', label='Percentil 90')

ax.set_title(f'Pronóstico hidrológico para los próximos meses en la subcuenca: {cuenca_sel}',fontweight='bold',fontsize=16)
ax.set_ylim(0,QC_chart_max*1.2)
#ax.set_ylim(0,500)
ax.legend(loc='best',title='Estado Hidrológico',fancybox=True, title_fontproperties={'weight':'bold'})
ax.set_ylabel('Discharge (m$^3$/s)');

ax.axvline(x = df_sel['fecha_pronostico'].iloc[0], color = 'r',label = 'now', linestyle=':', linewidth=2, mfc='white', mec='k');

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))   #to get a tick every 1 month
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))   #optional formatting 
plt.xticks(rotation=0);
plt.show()